# About
This point of this notebook is to properly format the namings in the catalogue & inventory.
(theres fuzzy differences in the strings)

Borrowings & inventory share the same titles, so: catalogue will attain these titles too.

But since the author names of catalogue are more reliable (fetched via an API) we will use them in the inventory.

In short:

Catalogue will get the titles of inventory,
Inventory will get the authors of catalogue

In [ ]:
# pandas for dataframe manipulation
import pandas as pd

# regex and sequence matcher to find similar fuzzy strings
import re
from difflib import SequenceMatcher

In [ ]:
# Read both CSV files
path1 = '../../data/processed/library_catalogue.csv'
path2 = '../../data/processed/library_inventory.csv'


# Read both files
catalogue = pd.read_csv(path1)
inventory = pd.read_csv(path2)

# Function to clean text for comparison
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    # Remove special characters, extra spaces, punctuation
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove common words that cause mismatch
    remove_words = ['professor', 'dr', 'mr', 'mrs', 'ms', 'jr', 'sr']
    for word in remove_words:
        text = text.replace(f' {word} ', ' ')
    return text

# Function to check similarity between two strings
def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

# Prepare cleaned versions for comparison
catalogue['title_clean'] = catalogue['Title'].apply(clean_text)
catalogue['author_clean'] = catalogue['Author'].apply(clean_text)
inventory['title_clean'] = inventory['Title'].apply(clean_text)
inventory['author_clean'] = inventory['Author'].apply(clean_text)

# Find and update matches
matches_found = 0

for idx, cat_row in catalogue.iterrows():
    cat_title = cat_row['title_clean']
    cat_author = cat_row['author_clean']
    
    best_match = None
    best_score = 0
    
    # Look for matching titles in inventory
    for _, inv_row in inventory.iterrows():
        inv_title = inv_row['title_clean']
        inv_author = inv_row['author_clean']
        
        # Calculate similarity scores
        title_similarity = similar(cat_title, inv_title)
        author_similarity = similar(cat_author, inv_author) if cat_author and inv_author else 0
        
        # Combined score (weight title more heavily)
        combined_score = title_similarity * 0.7 + author_similarity * 0.3
        
        # If we have a good match
        if combined_score > 0.7 and combined_score > best_score:
            best_score = combined_score
            best_match = inv_row
    
    # If we found a match, update the catalogue
    if best_match is not None and best_score > 0.7:
        # Get the original (not cleaned) values from inventory
        original_inv_title = best_match['Title']
        original_inv_author = best_match['Author']
        
        # Update catalogue with inventory values
        catalogue.at[idx, 'Title'] = original_inv_title
        catalogue.at[idx, 'Author'] = original_inv_author
        matches_found += 1
        
        print(f"Match found (score: {best_score:.2f}):")
        print(f"  Catalogue: {cat_row['Title'][:50]}... | {cat_row['Author']}")
        print(f"  Inventory: {original_inv_title[:50]}... | {original_inv_author}")
        print("-" * 80)

# Remove temporary columns
catalogue = catalogue.drop(columns=['title_clean', 'author_clean'])

# Save the updated catalogue
catalogue.to_csv('catalogue_updated.csv', index=False)

print(f"\nTotal matches found and updated: {matches_found}")
print(f"Updated file saved as: catalogue_updated.csv")
print(f"\nSample of updated records:")
print(catalogue[['Title', 'Author']].head(10).to_string())

Authors only in file1.csv:
  - A. R. WEBB
  - A. V. AHO
  - ABRAHAM SILBERSCHATZ
  - ADRIAN WALLWORK
  - ALAIN CARDON
  - ALAIN GIBAUD (ENSEIGNANT-CHERCHEUR EN INFORMATIQUE).)
  - ALAIN PILLER
  - ALAIN YGER
  - ALAN DENNIS
  - ALAN MEYERS
  - ALBERT
  - ALBERT (CHUN-CHEN) LIU
  - ALBERT EINSTEIN
  - ALBERT LEVINE
  - ALBERT PAUL MALVINO
  - ALEXANDRE CASAMAYOU-BOUCAU
  - ALFRED AHO
  - AMINE NAÏT-ALI
  - AMMAR ATTOUI
  - AMÉLIE BOUCHER
  - ANANY LEVITIN
  - ANDRIY BURKOV
  - ANDRÉ BAUMY
  - ANNE DENMAT
  - ANNE TASSO
  - ANTHONY JUTON
  - ARNAUD LEGRAND
  - ARQUES
  - AURÉLIEN GÉRON
  - AUZIMOUR
  - BENOÎT CHARROUX
  - BERNARD BONIN
  - BERNARD SARAMITO
  - BHUVAN UNHELKAR
  - BILL AULET
  - BILL MASCULL
  - BOI FALTINGS
  - BOSWELL/WILLIAM
  - BRUNO AEBISCHER
  - BRUNO BAYNAT
  - BRUNO MACIAS
  - BRUNO PETAZZONI
  - BUYENS
  - BÉNÉDICTE HUDAULT
  - CENTRE TESNIÈRE
  - CHARU C. AGGARWAL
  - CHRIS DATE
  - CHRISTIAN DUDOUET
  - CHRISTIAN GAUTIER
  - CHRISTIAN JACQUEMIN
  - CHRISTIAN JO